# 30 - Evaluate, compare, and report

The last run. Evaluates every completed final run that has no metrics artifact
yet, aggregates baseline/tuned results across seeds, writes the comparison
tables and figures, and - only when both publishing flags are set - publishes
the validated lightweight bundles.

Runs after the four model notebooks. A model that has not finished is reported
as missing rather than silently omitted; the cross-model comparison needs at
least two completed models and reports itself unavailable below that.


In [ ]:
DATASET_TRACK = "2class"
# True evaluates any completed run that has no metrics artifact yet.
EVALUATE_MISSING = True
# Publishing pushes to a public repository. It requires both flags:
# PUBLISH_RESULTS = True and DRY_RUN = False.
PUBLISH_RESULTS = False
DRY_RUN = True
# False runs against session storage, which is DELETED when the
# session ends. Only for smoke runs - never HPO or final training.
USE_GOOGLE_DRIVE = True


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git"
REPOSITORY_BRANCH = "main"
SMOKE_TEST = os.environ.get("SMOKE_TEST", "").lower() in {"1", "true", "yes"}

# The only logic a notebook still owns: make `src` importable. Everything after
# this line - Git state, platform detection, paths, dependency policy - lives in
# src/notebook_bootstrap.py so all notebooks behave identically.
_override = os.environ.get("BENCHMARK_REPO_ROOT")
_candidates = (
    [Path(_override).expanduser()]
    if _override
    else [
        Path.cwd(),
        *Path.cwd().parents,
        Path("/content/aerial-object-detection-benchmark"),
        Path("/kaggle/working/aerial-object-detection-benchmark"),
    ]
)
REPO_PATH = next(
    (
        candidate.resolve()
        for candidate in _candidates
        if (candidate / "src" / "notebook_bootstrap.py").is_file()
    ),
    None,
)
if REPO_PATH is None:
    _host = (
        Path("/content")
        if Path("/content").is_dir()
        else Path("/kaggle/working")
        if Path("/kaggle/working").is_dir()
        else None
    )
    if _host is None:
        raise RuntimeError(
            "Run this notebook from the repository, or set BENCHMARK_REPO_ROOT "
            "to an existing clone."
        )
    REPO_PATH = (_host / "aerial-object-detection-benchmark").resolve()
    REPO_PATH.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(REPO_PATH)],
        check=True,
    )
sys.path.insert(0, str(REPO_PATH))

from src.notebook_bootstrap import bootstrap_notebook

bootstrap = bootstrap_notebook(
    REPO_PATH,
    requirements_file='requirements-hpo-colab.txt',
    use_google_drive=USE_GOOGLE_DRIVE,
    smoke_test=SMOKE_TEST,
)
notebook_environment = bootstrap.environment
REPO_PATH = notebook_environment.repository_root
DRIVE_ROOT = notebook_environment.artifact_root
NOTEBOOK_PLATFORM = notebook_environment.platform
print(bootstrap.summary())


In [ ]:
from src.workflows.reporting import build_benchmark_report

result = build_benchmark_report(
    REPO_PATH,
    DRIVE_ROOT,
    DATASET_TRACK,
    # A smoke run reaches the real report but never launches an evaluation
    # subprocess or a publish, whatever the parameter cell above says.
    evaluate_missing=EVALUATE_MISSING and not SMOKE_TEST,
    publish=PUBLISH_RESULTS and not SMOKE_TEST,
    dry_run=DRY_RUN,
)
result
